# DINOv2 Frozen Probe — Contrail Segmentation

Baseline: fully frozen DINOv2 backbone, MLP head trained on top of intermediate features.
No VPT — backbone weights are never updated.

**Preprocessing notes:**
- `mask_only=True` → 3-channel false-color image at frame 4 (the labeled frame)
- DINOv2 normalization applied via `AutoImageProcessor`

## 1. Check GPU

In [ ]:
!nvidia-smi

In [ ]:
import os, subprocess, sys, re

if not os.path.exists('/kaggle/working/contrail-segmentation'):
    os.system('git clone https://github.com/aryangarg794/contrail-segmentation.git /kaggle/working/contrail-segmentation')
os.chdir('/kaggle/working/contrail-segmentation')
os.system('git checkout attention-unet-training && git pull')

result = subprocess.run([sys.executable, '-m', 'pip', 'show', 'torch'],
                        capture_output=True, text=True)
m = re.search(r'Version: (.+)', result.stdout)
current_ver = m.group(1).strip() if m else ''

if current_ver.startswith('2.4.0'):
    print(f'torch {current_ver} already installed — skipping restart')
else:
    print(f'torch {current_ver} detected — installing 2.4.0+cu121...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '--no-user', '--force-reinstall',
                    'torch==2.4.0', 'torchvision==0.19.0',
                    '--index-url', 'https://download.pytorch.org/whl/cu121'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '--no-user', '--force-reinstall', 'triton==2.3.1'], check=True)
    print('Done — restarting kernel...')
    os._exit(0)

In [ ]:
# Run this cell AFTER the kernel restarts (skip the torch cell above)
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy>=2.1', 'scipy>=1.15.0',
                'lightning', 'transformers>=5.3.0', 'timm',
                'albumentations',
                'segmentation-models-pytorch',
                'hydra-core', 'wandb', 'dill', 'rich', 'tqdm'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)

import importlib
sys.path.insert(0, '/kaggle/working/contrail-segmentation/src')
importlib.invalidate_caches()

import torch
print(f'torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
x = torch.tensor([1.0]).cuda()
print(f'CUDA test: PASSED {x}')

## 2. Dataset setup

Add datasets via the Kaggle sidebar:
- `nicolashornea/clean-train`
- `nicolashornea/train-metadata`

In [ ]:
import os

TRAIN_DIR = '/kaggle/input/clean-train/train'
META_PATH = '/kaggle/input/train-metadata/train_metadata.csv'
print('Train dir exists:', os.path.exists(TRAIN_DIR))
print('Metadata exists: ', os.path.exists(META_PATH))

## 3. Patch data paths at runtime

In [ ]:
import os, pandas as pd
import contrail_segmentation.data.utils as data_utils

data_utils.DATA_DIR  = TRAIN_DIR
data_utils.META_PATH = META_PATH

metadata  = pd.read_csv(META_PATH)
available = set(os.listdir(TRAIN_DIR))
metadata  = metadata[metadata['record_id'].apply(lambda x: str(int(x))).isin(available)].reset_index(drop=True)
data_utils.metadata = metadata

print(f'Dataset size: {len(data_utils.metadata)} records ({len(available)} on disk)')

## 4. W&B login

In [ ]:
import wandb
from kaggle_secrets import UserSecretsClient

wandb.login(key=UserSecretsClient().get_secret('wandb'))

## 5. Build preprocessing transforms

In [ ]:
import albumentations as A
from transformers import AutoImageProcessor
from huggingface_hub import login

login(token=UserSecretsClient().get_secret('hug'))

MODEL_NAME = 'facebook/dinov2-large'

processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
mean, std = processor.image_mean, processor.image_std
print(f'DINOv2 norm  mean={mean}  std={std}')

train_transform = A.Compose([
    A.ShiftScaleRotate(scale_limit=0.2, rotate_limit=180, shift_limit=0.3,
                       border_mode=0, value=0, p=0.5),
    A.HorizontalFlip(),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.3, p=0.5),
    A.Normalize(mean=mean, std=std),
])

val_transform = A.Compose([
    A.Normalize(mean=mean, std=std),
])

## 6. Build dataloaders

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from contrail_segmentation.data.dataset import ContrailDataset

SEED        = 0
BATCH_SIZE  = 16
NUM_WORKERS = 2

torch.manual_seed(SEED)
np.random.seed(SEED)
generator = torch.Generator().manual_seed(SEED)

full_dataset = ContrailDataset(mask_only=True)
indices      = np.arange(len(full_dataset))
np.random.shuffle(indices)
train_size   = int(0.8 * len(indices))
train_idx, val_idx = indices[:train_size], indices[train_size:]

train_set = ContrailDataset(mask_only=True, transform=train_transform)
val_set   = ContrailDataset(mask_only=True, transform=val_transform)

train_loader = DataLoader(Subset(train_set, train_idx), batch_size=BATCH_SIZE,
                          shuffle=True, generator=generator,
                          pin_memory=True, num_workers=NUM_WORKERS)
val_loader   = DataLoader(Subset(val_set, val_idx), batch_size=BATCH_SIZE,
                          pin_memory=True, num_workers=NUM_WORKERS)

print(f'Train: {len(train_idx)} | Val: {len(val_idx)}')

## 7. Build model

In [ ]:
from contrail_segmentation.models.dino_probe import DINOv2Probe

model = DINOv2Probe(
    model_name=MODEL_NAME,
    lr=1e-4,
    wd=1e-3,
    threshold=0.5,
    tversky_alpha=0.3,
    tversky_beta=0.7,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} params ({100*trainable/total:.1f}%)')

## 8. Train

In [ ]:
from datetime import datetime
from lightning.pytorch import Trainer
from lightning.pytorch.loggers import WandbLogger
from contrail_segmentation.train.utils import find_best_threshold

MAX_EPOCHS = 100
timestamp  = datetime.now().strftime('%d_%b_%Y__%Hh%Mm')
run_name   = f'dinov2_probe_seed{SEED}_{timestamp}'

logger = WandbLogger(project='contrail-segmentation', name=run_name, save_dir='wandb_logs')

trainer = Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator='gpu',
    devices=1,
    precision='16-mixed',
    log_every_n_steps=1,
    logger=logger,
)

trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

## 9. Test

In [ ]:
best_thresh = find_best_threshold(model, val_loader)
model.threshold = best_thresh
model.mask_only = True

test_metrics = trainer.test(model, dataloaders=val_loader)
print(test_metrics)

## 10. Save checkpoint

In [ ]:
torch.save(model.state_dict(), f'dinov2_probe_{timestamp}.pt')
print('Saved.')